# Uniform semi-infinite PIKAN prediction

Train a separate semi-infinite PIKAN with uniform interior samples and uniform top-boundary samples, then evaluate and save its checkpoint.

In [1]:
import numpy as np

In [ ]:
from pathlib import Path
import sys
from importlib import reload
import matplotlib.pyplot as plt
import numpy as np
import torch

notebook_dir = Path.cwd().resolve()
repo_root = next((path for path in [notebook_dir, *notebook_dir.parents] if (path / 'utils').is_dir()), notebook_dir)
utilities_dir = repo_root / 'utils'
if str(utilities_dir) not in sys.path:
    sys.path.insert(0, str(utilities_dir))
import pinns_infinite
import pinns_semi_infinite
import semi_infinite
reload(semi_infinite)
reload(pinns_infinite)
reload(pinns_semi_infinite)
from pinns_semi_infinite import build_models_KAN, set_seed, train_dual_network_semi_inf
from semi_infinite import analytical_solution_semi_inf, coefficient_semi_inf, evaluate_model_semi_inf
set_seed(42)
torch.set_default_dtype(torch.float32)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [ ]:
import pandas as pd

optimization_dir = (
    repo_root
    / "main"
    / "02_hyperparameter_tunning"
    / "results_kan_semi_infinite_optuna_2026-09-19_19-44-28"
)
study_path = optimization_dir / "data" / "study.csv"
study = pd.read_csv(study_path)
study = study[study["state"] == "COMPLETE"]
best_trial = study.loc[study["value"].idxmin()]

# Reuse the optimal semi-infinite hyperparameters, but train with uniform sampling.
config = {
    "mode": "train",
    "checkpoint_name": "pikan_semi_infinite_uniform_optimal_config_weights.pt",
    "hidden_layers": int(best_trial["params_hidden_layers"]),
    "hidden_units": int(best_trial["params_hidden_units"]),
    "grid_size": int(best_trial["params_grid_size"]),
    "spline_order": int(best_trial["params_spline_order"]),
    "adam_lr": float(best_trial["params_learning_rate"]),
    "adam_iters": 2000,
    "lbfgs_iters": 2000,
    "sampling": "uniform",
    "sigma": 5.5,
    "exp_scale": 4.0,
    "n_obs_u": 100,
    "n_boundary_u": 100,
    "n_obs_k": 100,
    "n_pde": 1000,
    "seed": 42,
    "pde_alpha": 0.5,
    "pde_beta": 5.0,
    "epsilon": 1.0,
    "train_domain": (-5.0, 5.0, -5.0, 0.0),
    "eval_domain": (-10.0, 10.0, -10.0, 0.0),
}

print("Optimal semi-infinite configuration extracted from hyperparameter optimization:")
print(f"Optimization trial number: {int(best_trial['number'])}")
print(f"Optimization mean global error: {best_trial['value']:.12e}")
print("Sampling overridden for this training: uniform")
for name, value in config.items():
    print(f"{name}: {value}")

mode: load
checkpoint_name: pikan_semi_infinite_uniform_weights.pt
hidden_layers: 3
hidden_units: 25
grid_size: 5
spline_order: 3
adam_lr: 0.001
adam_iters: 2000
lbfgs_iters: 2000
sampling: uniform
sigma: 5.5
exp_scale: 4.0
n_obs_u: 100
n_boundary_u: 100
n_obs_k: 100
n_pde: 1000
seed: 2
pde_alpha: 0.5
pde_beta: 5.0
epsilon: 1.0
train_domain: (-5.0, 5.0, -5.0, 0.0)
eval_domain: (-10.0, 10.0, -10.0, 0.0)


In [ ]:
model_u, model_k = build_models_KAN(
    device=device,
    hidden_layers=config['hidden_layers'],
    hidden_units=config['hidden_units'],
    grid_size=config['grid_size'],
    spline_order=config['spline_order'],
)

results_dir = repo_root / 'main' / '03_individual_prediction' / 'results'
available_checkpoints = sorted(results_dir.glob('*.pt'))
print('Available stored models:')
for checkpoint in available_checkpoints:
    print(f'  - {checkpoint.name}')

mode = config['mode'].lower()
if mode not in {'train', 'load'}:
    raise ValueError("config['mode'] must be either 'train' or 'load'")

if mode == 'load':
    weights_path = results_dir / config['checkpoint_name']
    if not weights_path.exists():
        raise FileNotFoundError(
            f"Stored checkpoint not found: {weights_path}. "
            f"Choose one of: {[path.name for path in available_checkpoints]}"
        )
    checkpoint = torch.load(weights_path, map_location=device)
    model_u.load_state_dict(checkpoint['model_u'])
    model_k.load_state_dict(checkpoint['model_k'])
    config.update(checkpoint.get('config', {}))
    metrics = checkpoint.get('metrics', {})
    print(f'Loaded stored model: {weights_path}')
else:
    history = train_dual_network_semi_inf(
        model_u,
        model_k,
        adam_lr=config['adam_lr'],
        adam_iters=config['adam_iters'],
        lbfgs_iters=config['lbfgs_iters'],
        verbose=True,
        print_every=100,
        save_every=100,
        lambda_pde_scheduler=True,
        adaptive_weights=True,
        alpha=7,
        update_every=100,
        regularization=False,
        sampling=config['sampling'],
        sigma=config['sigma'],
        exp_scale=config['exp_scale'],
        n_obs_u=config['n_obs_u'],
        n_boundary_u=config['n_boundary_u'],
        n_obs_k=config['n_obs_k'],
        n_pde=config['n_pde'],
        seed=config['seed'],
        save_results=True,
        base_dir=str(results_dir),
        run_name='pikan_semi_infinite_uniform',
        pde_alpha=config['pde_alpha'],
        pde_beta=config['pde_beta'],
        epsilon=config['epsilon'],
        device=device,
    )
    weights_path = results_dir / config['checkpoint_name']

model_u.eval()
model_k.eval()


if mode == 'train':
    torch.save(
        {
            'model_u': model_u.state_dict(),
            'model_k': model_k.state_dict(),
            'config': config,
            'metrics': metrics,
        },
        weights_path,
    )
    print(f'Saved uniform checkpoint to: {weights_path}')

Available stored models:
  - pikan_infinite_tuned_weights.pt
  - pikan_semi_infinite_tuned_weights.pt
  - pikan_semi_infinite_uniform_weights.pt
Loaded stored model: /home/orincon/unbounded-domains/main/03_individual_prediction/results/pikan_semi_infinite_uniform_weights.pt


KAN(
  (layers): ModuleList(
    (0-3): 4 x KANLinear(
      (base_activation): SiLU()
    )
  )
)

In [ ]:
evaluation = evaluate_model_semi_inf(
    model_u=model_u,
    model_k=model_k,
    analytical_solution=analytical_solution_semi_inf,
    coefficient=coefficient_semi_inf,
    train_domain=config["train_domain"],
    eval_domain=config["eval_domain"],
    n_grid=400,
    alpha=config["pde_alpha"],
    beta=config["pde_beta"],
    epsilon=config["epsilon"],
    device=device,
    verbose=True,
)

metric_names = [
    "err_u_global",
    "err_k_global",
    "err_u_inside",
    "err_k_inside",
    "err_u_outside",
    "err_k_outside",
]

metrics = {
    name: float(evaluation[name])
    for name in metric_names
}

# Mean global error
metrics["err_mean_global"] = (
    metrics["err_u_global"] +
    metrics["err_k_global"]
) / 2.0

metrics

Semi-infinite-domain spatial generalization (MAE)
u (global): 2.890e-02
k (global): 1.577e-01
u (inside): 3.997e-03
k (inside): 1.250e-03
u (outside): 3.720e-02
k (outside): 2.098e-01


{'err_u_global': 0.028901798557838556,
 'err_k_global': 0.1576793934715561,
 'err_u_inside': 0.003996963691537962,
 'err_k_inside': 0.0012496225980278691,
 'err_u_outside': 0.03720341017993875,
 'err_k_outside': 0.20982265042939882}